# 📓 PySpark + Jupyter: An Intermediate Example

This notebook builds on
[`pyspark_jupyter_example.ipynb`](./pyspark_jupyter_example.ipynb) (a
single DataFrame + one filter) with a few more everyday PySpark moves,
mirroring [`../marimo/pyspark_marimo_intermediate.py`](../marimo/pyspark_marimo_intermediate.py)
so you can compare the two side by side:

* joining two DataFrames
* `groupBy().agg()` with multiple aggregates, then `orderBy()`
* a derived column with `when()` / `otherwise()`
* filtering by two conditions at once (department *and* salary)

**Setup:**
```bash
pip install jupyter pyspark
jupyter notebook pyspark_jupyter_intermediate.ipynb
```

As with the basic example, this notebook is **not reactive**: change
`DEPT_FILTER` or `MIN_SALARY` below and you have to manually re-run
that cell (and anything after it) — the marimo version re-runs
automatically when you move its widgets.

## Step 1 — Start a SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, max as spark_max, when

spark = (
    SparkSession.builder
    .appName("pyspark-jupyter-intermediate")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## Step 2 — Create Two DataFrames and Join Them

`employees` only knows each person's `dept_id`; `departments` maps
that id to a human-readable name. Joining them is the usual way to
bring related data together in Spark.

In [2]:
some_employees = [
    (1, "alex", 10, 12000),
    (2, "jane", 20, 45000),
    (3, "rafa", 10, 56000),
    (4, "ted",  20, 145000),
    (5, "xo2",  30, 1332000),
    (6, "mary", 30, 555000),
    (7, "coco", 10, 33000),
    (8, "lee",  20, 78000),
]
employees = spark.createDataFrame(
    some_employees, ["id", "name", "dept_id", "salary"]
)

some_departments = [
    (10, "Engineering"),
    (20, "Sales"),
    (30, "Executive"),
]
departments = spark.createDataFrame(some_departments, ["dept_id", "dept_name"])

joined = employees.join(departments, on="dept_id", how="inner")
joined.show()

+-------+---+----+-------+-----------+
|dept_id| id|name| salary|  dept_name|
+-------+---+----+-------+-----------+
|     10|  1|alex|  12000|Engineering|
|     10|  3|rafa|  56000|Engineering|
|     10|  7|coco|  33000|Engineering|
|     20|  2|jane|  45000|      Sales|
|     20|  4| ted| 145000|      Sales|
|     20|  8| lee|  78000|      Sales|
|     30|  5| xo2|1332000|  Executive|
|     30|  6|mary| 555000|  Executive|
+-------+---+----+-------+-----------+



## Step 3 — Add a Derived Column with `when()` / `otherwise()`

`when()`/`otherwise()` is PySpark's inline if/else for building a new
column out of existing ones — here, bucketing salary into bands
without writing a UDF (see [`../UDF/`](../UDF) for the UDF way to do
this kind of thing).

In [3]:
banded = joined.withColumn(
    "salary_band",
    when(col("salary") < 50_000, "Low")
    .when(col("salary") < 200_000, "Medium")
    .otherwise("High"),
)
banded.show()

+-------+---+----+-------+-----------+-----------+
|dept_id| id|name| salary|  dept_name|salary_band|
+-------+---+----+-------+-----------+-----------+
|     10|  1|alex|  12000|Engineering|        Low|
|     10|  3|rafa|  56000|Engineering|     Medium|
|     10|  7|coco|  33000|Engineering|        Low|
|     20|  2|jane|  45000|      Sales|        Low|
|     20|  4| ted| 145000|      Sales|     Medium|
|     20|  8| lee|  78000|      Sales|     Medium|
|     30|  5| xo2|1332000|  Executive|       High|
|     30|  6|mary| 555000|  Executive|       High|
+-------+---+----+-------+-----------+-----------+



## Step 4 — `groupBy().agg()` and `orderBy()`

A per-department summary: headcount, average salary, and max salary,
sorted by average salary descending.

In [4]:
summary = (
    joined.groupBy("dept_name")
    .agg(
        count("*").alias("num_employees"),
        avg("salary").alias("avg_salary"),
        spark_max("salary").alias("max_salary"),
    )
    .orderBy("avg_salary", ascending=False)
)
summary.show()

+-----------+-------------+------------------+----------+
|  dept_name|num_employees|        avg_salary|max_salary|
+-----------+-------------+------------------+----------+
|  Executive|            2|          943500.0|   1332000|
|      Sales|            3| 89333.33333333333|    145000|
|Engineering|            3|33666.666666666664|     56000|
+-----------+-------------+------------------+----------+



## Step 5 — Filter by Two Conditions

Change `DEPT_FILTER` (use `None` for all departments) and `MIN_SALARY`
below, then re-run this cell.

In [5]:
DEPT_FILTER = "Engineering"   # or None for all departments
MIN_SALARY = 30_000

filtered = banded.filter(col("salary") >= MIN_SALARY)
if DEPT_FILTER is not None:
    filtered = filtered.filter(col("dept_name") == DEPT_FILTER)
filtered.show()

+-------+---+----+------+-----------+-----------+
|dept_id| id|name|salary|  dept_name|salary_band|
+-------+---+----+------+-----------+-----------+
|     10|  3|rafa| 56000|Engineering|     Medium|
|     10|  7|coco| 33000|Engineering|        Low|
+-------+---+----+------+-----------+-----------+



## Step 6 — Stop Spark

In [6]:
spark.stop()